## Imports

In [2]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
import sys
import matplotlib.pyplot as plt
from ipywidgets import interact


from data import *
from anser import *

## Generate 8 coil field generator

In [3]:
coils_global = build_field_generator(N_turns,l,w,s,z_thick,centres,rotations)

## Backward function ( PO -> flux )

In [3]:
def backward(x,y,z,theta,phi, coils_global = coils_global):
    
    sensor_normal = np.array([np.sin(theta)*np.cos(phi),
                              np.sin(theta)*np.sin(phi),
                              np.cos(theta)]) 
    
    h = field_coil_calc(1,coils_global,np.array([x,y,z]))

    flux = np.sum(sensor_normal*h, axis = -1)

    return flux

In [4]:
backward(0,0,0,0,0)

array([ 3.57208257, 37.02659641,  3.57480851, 36.88407379, 37.03395184,
        3.56676596, 37.04083628,  3.57623473])

In [5]:
forward_model(np.array([0,0,0,0,0]),coils_global)

array([ 3.57208257, 37.02659641,  3.57480851, 36.88407379, 37.03395184,
        3.56676596, 37.04083628,  3.57623473])

## Translational stuff

First show the mirrorness between coils in x and y, look at coil pairs (0,7), (2,5) etc..

In [6]:
xs = np.linspace(-0.15,0.15,100)
ys = np.linspace(-0.15,0.15,100)
@interact(z=(0, 0.25, 0.01), theta=(0,np.pi, np.pi/10), phi=(0,2*np.pi, np.pi/10))
def plot_flux_xy(z, theta, phi):
    
    fluxes = np.zeros((len(xs), len(ys), 8))
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            fluxes[i, j] = forward_model(np.array([x,y,z,theta,phi]),coils_global)
    vmax = np.abs(fluxes).max()  # shared color scale across all 8 coils

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    for coil, ax in enumerate(axes.flat):
        im = ax.imshow(fluxes[:, :, coil], origin='lower',
                        extent=[xs[0], xs[-1], ys[0], ys[-1]],
                        aspect='auto', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Coil {coil}')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        fig.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

    
    
    flux_mag = np.linalg.norm(fluxes, axis=-1)
    
    plt.figure(figsize=(8, 6))
    im = plt.imshow(flux_mag, origin='lower',
                    extent=[xs[0], xs[-1], ys[0], ys[-1]],
                    aspect='auto')
    plt.colorbar(im, label='Total flux magnitude')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Combined sensitivity across orientation')
    plt.show()

interactive(children=(FloatSlider(value=0.12, description='z', max=0.25, step=0.01), FloatSlider(value=1.57079…

Now do sweeps varying one parameter and plotting each flux and the flux magnitude

### Varying x

In [7]:
xs = np.linspace(-0.15,0.15,100)
@interact(y=(-0.15, 0.15, 0.01), z=(0, 0.25, 0.01), theta=(0,np.pi,np.pi/10), phi=(0,2*np.pi,np.pi/20))
def plot_flux_trans_x(y, z, theta,phi):
    
    fluxes = np.zeros((len(xs), 8))
    for i, x in enumerate(xs):
        fluxes[i] = forward_model(np.array([x,y,z,theta,phi]),coils_global)

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    for coil, ax in enumerate(axes.flat):
        ax.plot(xs,fluxes[:,coil])
        ax.set_title(f'Coil {coil}')
        ax.set_xlabel('x')
        ax.set_ylabel('flux')
    plt.tight_layout()
    plt.show()

    flux_mag = np.linalg.norm(fluxes, axis=-1)
    plt.figure(figsize=(8, 6))
    plt.xlabel('x')
    plt.ylabel('flux magnitude')
    plt.title('Flux magnitude varying x')
    plt.plot(xs,flux_mag)
    plt.show()

    
    


interactive(children=(FloatSlider(value=0.0, description='y', max=0.15, min=-0.15, step=0.01), FloatSlider(val…

### Varying z

In [8]:
zs = np.linspace(0,0.25,100)
@interact(x=(-0.15, 0.15, 0.01), y=(-0.15, 0.15, 0.01), theta=(0,np.pi,np.pi/10), phi=(0,2*np.pi,np.pi/20))
def plot_flux_trans_x(x, y, theta,phi):
    
    fluxes = np.zeros((len(xs), 8))
    for i, z in enumerate(zs):
        fluxes[i] = forward_model(np.array([x,y,z,theta,phi]),coils_global)

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    for coil, ax in enumerate(axes.flat):
        ax.plot(xs,fluxes[:,coil])
        ax.set_title(f'Coil {coil}')
        ax.set_xlabel('z')
        ax.set_ylabel('flux')
    plt.tight_layout()
    plt.show()

    flux_mag = np.linalg.norm(fluxes, axis=-1)
    plt.figure(figsize=(8, 6))
    plt.xlabel('z')
    plt.ylabel('flux magnitude')
    plt.title('Flux magnitude varying z')
    plt.plot(xs,flux_mag)
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='x', max=0.15, min=-0.15, step=0.01), FloatSlider(val…

# Rotational stuff

In [5]:
phis = np.linspace(0,2*np.pi,100)
thetas = np.linspace(0,np.pi,100)
coils = [f'Coil {i}' for i in range(coils_global.shape[0])]

In [10]:
@interact(x=(-0.15, 0.15, 0.01), y=(-0.15, 0.15, 0.01), z=(0,0.25,0.01))
def plot_flux_angle(x, y, z):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    for i in range(coils_global.shape[0]):
        ax.plot3D(coils_global[i, :, 0], coils_global[i, :, 1], coils_global[i, :, 2],
                   linewidth=0.5, label=coils[i])

    # sensor position
    ax.scatter(x, y, z, color='black', s=50, label='sensor')

    ax.legend()
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_zlabel('z (m)')
    plt.show()

    
    fluxes = np.zeros((len(thetas), len(phis), 8))
    for i, t in enumerate(thetas):
        for j, p in enumerate(phis):
            fluxes[i, j] = forward_model(np.array([x,y,z,theta,phi]),coils_global)

    vmax = np.abs(fluxes).max()  # shared color scale across all 8 coils

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    for coil, ax in enumerate(axes.flat):
        im = ax.imshow(fluxes[:, :, coil], origin='lower',
                        extent=[phis[0], phis[-1], thetas[0], thetas[-1]],
                        aspect='auto', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Coil {coil}')
        ax.set_xlabel('phi')
        ax.set_ylabel('theta')
        fig.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

    flux_mag = np.linalg.norm(fluxes, axis=-1)
    plt.figure(figsize=(8, 6))
    im = plt.imshow(flux_mag, origin='lower',
                    extent=[phis[0], phis[-1], thetas[0], thetas[-1]],
                    aspect='auto', cmap='viridis')
    plt.colorbar(im, label='Total flux magnitude')
    plt.xlabel('phi')
    plt.ylabel('theta')
    plt.title('Combined sensitivity across orientation')
    plt.show()
    
    

interactive(children=(FloatSlider(value=0.0, description='x', max=0.15, min=-0.15, step=0.01), FloatSlider(val…

## Correlation heatmap

In [11]:
@interact(x=(-0.15, 0.15, 0.01), y=(-0.15, 0.15, 0.01), z=(0,0.25,0.01))
def plot_flux_angle_heatmap(x, y, z):

    fluxes = np.zeros((len(thetas), len(phis), 8))
    for i, t in enumerate(thetas):
        for j, p in enumerate(phis):
            fluxes[i, j] = forward_model(np.array([x,y,z,t,p]),coils_global)

    flux_flat = fluxes.reshape(-1, 8)        # (10000, 8) — each row is one (theta,phi) sample
    corr = np.corrcoef(flux_flat, rowvar=False)  # (8, 8)
    
    plt.figure(figsize=(6, 6))
    im = plt.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
    plt.colorbar(im, label='Correlation')
    plt.xticks(range(8), coils, rotation=45)
    plt.yticks(range(8), coils)
    plt.title('Coil flux correlation across (theta, phi) sweep')
    plt.tight_layout()
    plt.show()


interactive(children=(FloatSlider(value=0.0, description='x', max=0.15, min=-0.15, step=0.01), FloatSlider(val…

In [7]:

@interact(z=(0, 0.25, 0.01), theta=(0, np.pi/2, np.pi/10), phi=(0, 2*np.pi, np.pi/10))
def plot_sensitivity(z, theta, phi):
    param_names = ['x', 'y', 'z', 'theta', 'phi']
    sens = np.zeros((len(xs), len(ys), 5))

    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            J = jacobian(x, y, z, theta, phi)
            for k in range(5):
                sens[i, j, k] = np.linalg.norm(J[:, k])

    fig, axes = plt.subplots(3, 2, figsize=(18, 8))
    for k, ax in enumerate(axes.flat):
        if k == 5: 
            ax.set_visible(False)
            break
        im = ax.imshow(sens[:, :, k], origin='lower',
                       extent=[xs[0], xs[-1], ys[0], ys[-1]],
                       aspect='equal', cmap='viridis')
        fig.colorbar(im, ax=ax)
        ax.set_title(f'dflux/d{param_names[k]}')
        ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.12, description='z', max=0.25, step=0.01), FloatSlider(value=0.62831…